# Spark — nettoyer, agréger, fenêtrer

**Formation Big Data — ANSD / Data Innovation Lab**

Le fichier de recensement porte les défauts habituels des données
administratives : libellés incohérents, âges impossibles, dates saisies dans
trois formats différents, doublons.

Nous allons le nettoyer, en tirer des tableaux, puis découvrir les **fonctions
de fenêtrage** — celles qui calculent une valeur sur un groupe sans écraser le
détail.

Le code est fourni : exécutez, observez, modifiez les valeurs pour explorer.

## 1. Session et données

In [ ]:
import os
import time
from pathlib import Path

from pyspark.sql import SparkSession, Window, functions as F

DONNEES = Path(os.environ.get("DONNEES", "/travail/donnees"))
FICHIER = DONNEES / "individus.csv"

spark = (
    SparkSession.builder
    .appName("nettoyer_agreger")
    .master("local[*]")             # local[2] si votre poste manque de mémoire
    .config("spark.sql.shuffle.partitions", "8")
    .config("spark.driver.memory", "2g")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("ERROR")

brut = spark.read.csv(str(FICHIER), header=True, inferSchema=True)
print("Spark", spark.version)
print(f"{brut.count():,} lignes".replace(",", " "))

In [ ]:
import pyspark

print("PySpark :", pyspark.__version__)
print("Spark   :", spark.version)

import sys
print(sys.executable)

## 2. Les fonctions de colonnes

En Spark, on ne manipule pas des valeurs mais on **décrit des colonnes**.
`F.col("region")` désigne la colonne entière, et `F.upper(...)` décrit une
transformation que le moteur appliquera lui-même.

Votre code Python n'est jamais exécuté ligne par ligne : il construit une
description, que l'optimiseur traduit.

In [ ]:
brut.select(
    F.col("region"),
    F.upper(F.col("region")).alias("en_majuscules"),
    F.length(F.col("region")).alias("longueur"),
    F.trim(F.col("region")).alias("sans_espaces"),
).show(5, truncate=False)

### Le catalogue

`pyspark.sql.functions` contient plusieurs centaines de fonctions. Avant
d'écrire la vôtre, cherchez si elle existe : c'est presque toujours le cas.

In [ ]:
fonctions = [n for n in dir(F) if not n.startswith("_")]
print(f"{len(fonctions)} fonctions disponibles\n")
print("Quelques familles :")
for prefixe in ["to_", "try_", "regexp", "array_", "date_"]:
    exemples = [n for n in fonctions if n.startswith(prefixe)][:6]
    print(f"  {prefixe:<10} {', '.join(exemples)}")

## 3. Le coût d'une fonction Python personnalisée

Une **UDF** permet d'écrire sa propre fonction Python. Chaque ligne sort alors
du moteur, passe en Python, puis revient. Mesurons.

In [ ]:
from pyspark.sql.types import IntegerType

echantillon = brut.select("region", "age").limit(200_000).cache()
echantillon.count()          # force la mise en cache

# Version native : décrite avec des fonctions de colonnes
groupe_natif = (
    F.when(F.col("age") < 15, 0)
     .when(F.col("age") < 35, 1)
     .when(F.col("age") < 60, 2)
     .otherwise(3)
)

# Version UDF : du Python, exécuté ligne par ligne
def groupe_python(age):
    if age is None:
        return None
    if age < 15:
        return 0
    if age < 35:
        return 1
    if age < 60:
        return 2
    return 3

groupe_udf = F.udf(groupe_python, IntegerType())

for libelle, expression in [("native", groupe_natif), ("UDF   ", groupe_udf(F.col("age")))]:
    depart = time.perf_counter()
    echantillon.withColumn("groupe", expression).groupBy("groupe").count().collect()
    print(f"  {libelle} : {time.perf_counter() - depart:5.2f} s")

L'écart est net. Et l'optimiseur ne voit plus à l'intérieur d'une UDF : il
ne peut ni pousser un filtre, ni fusionner des étapes.

**La règle** : chercher la fonction native d'abord, l'UDF en dernier recours.

In [ ]:
# Un effet de bord révélateur : ici, on ajoute la colonne UDF puis on fait un
# simple count() — qui n'a pas besoin de cette colonne.
depart = time.perf_counter()
echantillon.withColumn("groupe", groupe_udf(F.col("age"))).count()
print(f"count() après UDF : {time.perf_counter() - depart:.2f} s")
print("→ l'optimiseur a supprimé la colonne : elle n'était utile à personne.")

echantillon.unpersist()

## 4. Nettoyer

### Les libellés de région

In [ ]:
print("Libellés distincts avant :", brut.select("region").distinct().count())

propre = brut.withColumn("region", F.initcap(F.trim(F.col("region"))))

print("Libellés distincts après :", propre.select("region").distinct().count())
propre.select("region").distinct().orderBy("region").show(20, truncate=False)

### Les valeurs aberrantes

Un âge de 999 ou de −1 est un code de non-réponse mal nettoyé. On le
**dénombre avant de l'écarter** : une valeur supprimée en silence est une
information perdue.

In [ ]:
aberrants = propre.filter(~F.col("age").between(0, 110))
print(f"Âges impossibles : {aberrants.count():,}".replace(",", " "))
aberrants.groupBy("age").count().orderBy(F.col("count").desc()).show(5)

propre = propre.withColumn(
    "age",
    F.when(F.col("age").between(0, 110), F.col("age")).otherwise(None),
)

### Les dates : trois formats dans un même fichier

Le fichier mélange `AAAA-MM-JJ`, `JJ/MM/AAAA` et `JJ-MM-AAAA`.

> ⚠️ **Spark 4 est strict.** Le mode ANSI est actif par défaut : `to_date` avec
> un format qui ne correspond pas **lève une erreur** au lieu de renvoyer une
> valeur manquante. C'est `try_to_date` qu'il faut employer pour tolérer les
> écarts.

In [ ]:
print("Mode ANSI :", spark.conf.get("spark.sql.ansi.enabled"))

propre = propre.withColumn(
    "date_naissance",
    F.coalesce(
        F.try_to_date(F.col("date_naissance"), "yyyy-MM-dd"),
        F.try_to_date(F.col("date_naissance"), "dd/MM/yyyy"),
        F.try_to_date(F.col("date_naissance"), "dd-MM-yyyy"),
    ),
)

echecs = propre.filter(F.col("date_naissance").isNull()).count()
print(f"Dates non converties : {echecs}")
propre.select("date_naissance").show(5)

`coalesce` retient la première tentative qui aboutit. On compte toujours les
échecs restants : s'il y en a, c'est qu'un quatrième format existe.

### Les valeurs manquantes

In [ ]:
total = propre.count()

manquants = propre.select([
    F.round(100 * F.sum(F.col(colonne).isNull().cast("int")) / total, 2).alias(colonne)
    for colonne in ["niveau_instruction", "secteur_activite", "nb_pieces",
                    "sait_lire_ecrire", "age"]
])
manquants.show()

Le secteur d'activité est très souvent absent — c'est normal : seules les
personnes occupées en ont un. Une valeur manquante n'est pas toujours une
erreur, elle est parfois une information.

### Les doublons

In [ ]:
apres_dedoublonnage = propre.dropDuplicates()
print(f"Doublons exacts : {total - apres_dedoublonnage.count():,}".replace(",", " "))

# Sur une clé métier : deux enregistrements pour un même identifiant
par_identifiant = propre.dropDuplicates(["id_individu"])
print(f"Identifiants en double : {total - par_identifiant.count():,}"
      .replace(",", " "))

propre = apres_dedoublonnage

> `dropDuplicates` provoque un brassage : toutes les lignes candidates
> doivent se retrouver au même endroit pour être comparées. C'est une opération
> coûteuse, à ne pas répéter inutilement.

In [ ]:
# Cette table va servir plusieurs fois : on la conserve en mémoire
propre = propre.cache()
print(f"{propre.count():,} lignes après nettoyage".replace(",", " "))

## 5. Agréger

`groupBy` accepte plusieurs colonnes, et `agg` produit plusieurs indicateurs en
une seule passe — ce qui évite de multiplier les brassages.

In [ ]:
(propre.groupBy("region")
       .agg(
           F.count("*").alias("effectif"),
           F.round(F.avg("age"), 1).alias("age_moyen"),
           F.expr("percentile_approx(age, 0.5)").alias("age_median"),
       )
       .orderBy(F.col("effectif").desc())
       .show(10))

### Les comptages conditionnels

`sum(when(condition, 1).otherwise(0))` remplace les tableaux croisés
habituels : on compte, dans un même passage, plusieurs sous-populations.

In [ ]:
actif = F.col("situation_activite").isin("Occupé", "Chômeur")

(propre.filter(F.col("age") >= 15)
       .groupBy("region", "milieu_residence")
       .agg(
           F.count("*").alias("population_15_plus"),
           F.sum(actif.cast("int")).alias("actifs"),
           F.sum(F.col("situation_activite").eqNullSafe("Chômeur").cast("int"))
            .alias("chomeurs"),
       )
       .withColumn("taux_activite",
                   F.round(100 * F.col("actifs") / F.col("population_15_plus"), 1))
       .withColumn("taux_chomage",
                   F.round(100 * F.col("chomeurs") / F.col("actifs"), 1))
       .orderBy("region", "milieu_residence")
       .show(10))

## 6. Les fonctions de fenêtrage

Une agrégation **réduit** : dix lignes deviennent une. Une fenêtre **conserve**
toutes les lignes en ajoutant une valeur calculée sur un groupe.

C'est ce qui permet de calculer une part relative sans perdre le détail.

In [ ]:
# Une table de travail : effectifs par région et milieu de résidence
effectifs = (
    propre.groupBy("region", "milieu_residence")
          .agg(F.count("*").alias("effectif"))
)

effectifs.orderBy("region").show(6)

### Une part relative, calculée sur la région

`Window.partitionBy("region")` définit le groupe sur lequel calculer. La somme
porte sur la région, mais chaque ligne est conservée.

In [ ]:
par_region = Window.partitionBy("region")

(effectifs.withColumn("total_region", F.sum("effectif").over(par_region))
          .withColumn("part",
                      F.round(100 * F.col("effectif") / F.col("total_region"), 1))
          .orderBy("region", F.col("part").desc())
          .show(8))

### Classer au sein d'un groupe

`rank`, `dense_rank` et `row_number` attribuent un rang. La fenêtre doit alors
être **ordonnée**.

In [ ]:
classement = Window.partitionBy("region").orderBy(F.col("effectif").desc())

(effectifs.withColumn("rang", F.rank().over(classement))
          .filter(F.col("rang") == 1)
          .orderBy(F.col("effectif").desc())
          .show(10))

### Cumuler, et comparer à la ligne précédente

`rowsBetween` fixe l'étendue de la fenêtre. `lag` regarde la ligne précédente —
c'est la base de tout calcul d'évolution.

In [ ]:
totaux = (propre.groupBy("region")
                .agg(F.count("*").alias("effectif"))
                .orderBy(F.col("effectif").desc()))

ordre = Window.orderBy(F.col("effectif").desc())
cumul = ordre.rowsBetween(Window.unboundedPreceding, Window.currentRow)

(totaux.withColumn("cumul", F.sum("effectif").over(cumul))
       .withColumn("part_cumulee",
                   F.round(100 * F.col("cumul")
                           / F.sum("effectif").over(Window.partitionBy()), 1))
       .withColumn("ecart_precedent",
                   F.col("effectif") - F.lag("effectif").over(ordre))
       .show(10))

La colonne `part_cumulee` répond à une question classique : combien de
régions concentrent la moitié de la population ?

> ⚠️ Une fenêtre **sans** `partitionBy` couvre toute la table, donc rassemble
> toutes les données au même endroit. Sur un gros volume, c'est à éviter — ici
> la table agrégée ne fait que quatorze lignes.

### Un exemple sur les données individuelles

Les fenêtres s'appliquent aussi au détail : ci-dessous, l'âge de chaque personne
comparé à la moyenne de son ménage.

In [ ]:
par_menage = Window.partitionBy("id_menage")

(propre.select("id_menage", "prenom", "nom", "age", "lien_chef_menage")
       .withColumn("taille_menage", F.count("*").over(par_menage))
       .withColumn("age_moyen_menage",
                   F.round(F.avg("age").over(par_menage), 1))
       .withColumn("ecart",
                   F.round(F.col("age") - F.col("age_moyen_menage"), 1))
       .filter(F.col("taille_menage").between(4, 6))
       .orderBy("id_menage")
       .show(10))

## 7. Ce qu'il faut retenir

- On **décrit des colonnes** avec `F.*` ; le moteur exécute. Une **UDF** Python
  fait sortir du moteur et coûte cher — chercher la fonction native d'abord.
- Spark 4 active le **mode ANSI** par défaut : une conversion qui échoue lève
  une erreur. Employer `try_to_date`, `try_cast` et leurs équivalents pour
  tolérer les écarts.
- Toujours **dénombrer avant d'écarter** : aberrations, échecs de conversion,
  doublons.
- Une `agg` avec plusieurs indicateurs vaut mieux que plusieurs agrégations :
  chacune est un brassage.
- Une **fenêtre** enrichit sans réduire. `partitionBy` définit le groupe,
  `orderBy` l'ordre, `rowsBetween` l'étendue.
- `cache()` sur une table relue plusieurs fois ; `unpersist()` ensuite.

In [ ]:
propre.unpersist()
spark.stop()
print("Session arrêtée.")